In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import TFBertModel
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Flatten, Layer, MultiHeadAttention
from tensorflow.keras.models import Sequential
from lime.lime_tabular import LimeTabularExplainer
from lime.lime_text import LimeTextExplainer
import shap
from collections import defaultdict

In [2]:
# Load and preprocess data
cali_housing_path = '../data/California_Houses.csv'
RANDOM_SEED = 492
cali_df = pd.read_csv(cali_housing_path)
y = cali_df['Median_House_Value']
X = cali_df.drop(columns=['Median_House_Value'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3]:
class FeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, embed_dim):
        super(FeatureTokenizer, self).__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim
        
        self.feature_weights = self.add_weight(shape=(num_features, embed_dim),
                                               initializer='random_normal',
                                               trainable=True)
        self.feature_bias = self.add_weight(shape=(num_features, embed_dim),
                                            initializer='random_normal',
                                            trainable=True)
        self.layer_norm = LayerNormalization()
    
    def call(self, inputs):
        embeddings = inputs[..., tf.newaxis] * self.feature_weights + self.feature_bias
        embeddings = self.layer_norm(embeddings)
        return embeddings


In [4]:
# Define TransformerBlock
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, attention_mask=None, training=False):
        attn_output = self.att(inputs, inputs, attention_mask=attention_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Define FTTransformer
class FTTransformer(tf.keras.Model):
    def __init__(self, dim, dim_out, depth, heads, ff_dim, mlp_hidden, attn_dropout=0.1, ff_dropout=0.1):
        super(FTTransformer, self).__init__()
        self.transformers = [TransformerBlock(dim, heads, ff_dim, rate=attn_dropout) for _ in range(depth)]
        self.flatten_transformer_output = tf.keras.layers.Flatten()
        self.mlp_layers = [Dense(size, activation=activation) for size, activation in mlp_hidden]
        self.output_layer = Dense(dim_out)
    
    def call(self, inputs, attention_mask=None, training=False):
        x = inputs
        for transformer in self.transformers:
            x = transformer(x, attention_mask=attention_mask, training=training)
        x = self.flatten_transformer_output(x)
        for mlp_layer in self.mlp_layers:
            x = mlp_layer(x)
        return self.output_layer(x)


In [5]:
class BERTMLP(tf.keras.Model):
    def __init__(self, embed_dim=20, dim_out=1, depth=2, heads=4, ff_dim=128, mlp_hidden=[(128, 'relu'), (64, 'relu')], hidden_dim=64, dropout_rate=0.1):
        super(BERTMLP, self).__init__()
        self.transformer = FTTransformer(embed_dim, dim_out, depth, heads, ff_dim, mlp_hidden)
        self.dense1 = Dense(hidden_dim, activation='relu')
        self.dropout1 = Dropout(dropout_rate)
        self.dense2 = Dense(hidden_dim, activation='relu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense3 = Dense(1, activation='linear')
        self.heads = heads

    def call(self, inputs, attention_mask=None, training=False):
        batch_size = tf.shape(inputs)[0]
        seq_length = tf.shape(inputs)[1]
        if attention_mask is None:
            attention_mask = tf.ones((batch_size, seq_length, seq_length), dtype=tf.float32)
        bert_output = self.transformer(inputs, attention_mask=attention_mask, training=training)
        x = self.dense1(bert_output)
        x = self.dropout1(x, training=training)
        x = self.dense2(x)
        x = self.dropout2(x, training=training)
        x = self.dense3(x)
        return x

In [ ]:
def explain_lime_custom(model, instance, train_data, num_features, embed_dim):
    
    # Generate feature names based on the number of features
    feature_names = [train_data.columns()]
    
    explainer = LimeTabularExplainer(
        training_data=train_data,
        feature_names=feature_names,
        mode='regression'
    )
    
    # Aggregate the embeddings for the instance
    instance_agg = aggregate_embeddings(instance, method='mean')
    
    # Create a wrapper for the model's predict function
    # def predict_wrapper(x):
    #     x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
    #     x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
    #     return model(x_3d, training=True).numpy()
    
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    
    explanation = explainer.explain_instance(
        instance_agg[0], 
        predict_wrapper, 
        num_features=num_features
    )
    
    return explanation


In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, MultiHeadAttention
from lime.lime_tabular import LimeTabularExplainer
import shap

class FeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, embed_dim):
        super(FeatureTokenizer, self).__init__()
        self.num_features = num_features
        self.embed_dim = embed_dim
        
        self.feature_weights = self.add_weight(shape=(num_features, embed_dim),
                                               initializer='random_normal',
                                               trainable=True)
        self.feature_bias = self.add_weight(shape=(num_features, embed_dim),
                                            initializer='random_normal',
                                            trainable=True)
        self.layer_norm = LayerNormalization()
    
    def call(self, inputs):
        embeddings = inputs[..., tf.newaxis] * self.feature_weights + self.feature_bias
        embeddings = self.layer_norm(embeddings)
        return embeddings

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, attention_mask=None, training=False):
        attn_output = self.att(inputs, inputs, attention_mask=attention_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

class FTTransformer(tf.keras.Model):
    def __init__(self, dim, dim_out, depth, heads, ff_dim, mlp_hidden, attn_dropout=0.1, ff_dropout=0.1):
        super(FTTransformer, self).__init__()
        self.transformers = [TransformerBlock(dim, heads, ff_dim, rate=attn_dropout) for _ in range(depth)]
        self.flatten_transformer_output = tf.keras.layers.Flatten()
        self.mlp_layers = [Dense(size, activation=activation) for size, activation in mlp_hidden]
        self.output_layer = Dense(dim_out)
    
    def call(self, inputs, attention_mask=None, training=False):
        x = inputs
        for transformer in self.transformers:
            x = transformer(x, attention_mask=attention_mask, training=training)
        x = self.flatten_transformer_output(x)
        for mlp_layer in self.mlp_layers:
            x = mlp_layer(x)
        return self.output_layer(x)

def aggregate_embeddings(data, method='mean'):
    if method == 'mean':
        return np.mean(data, axis=-1)
    else:
        raise ValueError("Unsupported aggregation method")

def explain_lime_custom(model, instance, train_data, num_features, embed_dim):
    train_data_agg = aggregate_embeddings(train_data.numpy(), method='mean')
    feature_names = [f'feature_{i}' for i in range(num_features)]
    explainer = LimeTabularExplainer(
        training_data=train_data_agg,
        feature_names=feature_names,
        mode='regression'
    )
    instance_agg = aggregate_embeddings(instance.numpy().reshape(1, num_features, embed_dim), method='mean')
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    explanation = explainer.explain_instance(
        instance_agg[0], 
        predict_wrapper, 
        num_features=num_features
    )
    return explanation

def explain_shap_custom(model, instance, train_data, num_features, embed_dim):
    train_data_agg = aggregate_embeddings(train_data.numpy(), method='mean')
    def predict_wrapper(x):
        x_expanded = np.expand_dims(x, -1).repeat(embed_dim, axis=-1)
        x_3d = tf.convert_to_tensor(x_expanded, dtype=tf.float32)
        pred = model(x_3d, training=True).numpy()
        return np.mean(pred, axis=0)
    explainer = shap.KernelExplainer(predict_wrapper, train_data_agg)
    instance_agg = aggregate_embeddings(instance.numpy().reshape(1, num_features, embed_dim), method='mean')
    shap_values = explainer.shap_values(instance_agg)
    return shap_values

class ModelWithUncertainty(tf.keras.Model):
    def __init__(self, input_dim, hidden_dim, dropout_rate=0.1, method='mc_dropout'):
        super(ModelWithUncertainty, self).__init__()
        self.method = method
        self.model = FTTransformer(dim=input_dim, dim_out=1, depth=2, heads=4, ff_dim=128, mlp_hidden=[(128, 'relu'), (64, 'relu')], attn_dropout=dropout_rate, ff_dropout=dropout_rate)
        self.tokenizer = FeatureTokenizer(input_dim, embed_dim=hidden_dim)
    
    def call(self, inputs, training=False):
        tokenized_inputs = self.tokenizer(inputs)
        return self.model(tokenized_inputs, training=training)
    
    def predict_with_explanation(self, inputs, train_data, num_samples=10):
        num_features = inputs.shape[1]
        embed_dim = inputs.shape[2]
        predictions = []
        lime_explanations = []
        shap_explanations = []
        for i in range(len(inputs)):
            instance = inputs[i:i+1]
            for _ in range(num_samples):
                prediction = self(inputs[i:i+1], training=True)
                predictions.append(prediction)
                lime_explanation = explain_lime_custom(self, instance, train_data, num_features, embed_dim)
                shap_explanation = explain_shap_custom(self, instance, train_data, num_features, embed_dim)
                lime_explanations.append(lime_explanation)
                shap_explanations.append(shap_explanation)
        predictions = tf.stack(predictions, axis=0)
        prediction_mean = tf.reduce_mean(predictions, axis=0)
        prediction_std = tf.math.reduce_std(predictions, axis=0)
        return prediction_mean.numpy(), prediction_std.numpy(), lime_explanations, shap_explanations

    def predict_with_uncertainty(self, inputs, num_samples=10):
        predictions = []
        for _ in range(num_samples):
            prediction = self(inputs, training=True)
            predictions.append(prediction)
        predictions = tf.stack(predictions, axis=0)
        prediction_mean = tf.reduce_mean(predictions, axis=0)
        prediction_std = tf.math.reduce_std(predictions, axis=0)
        return prediction_mean.numpy(), prediction_std.numpy()

# Example usage:
# model = ModelWithUncertainty(input_dim=X_train.shape[1], hidden_dim=20, dropout_rate=0.1)
# predictions, uncertainties, lime_explanations, shap_explanations = model.predict_with_explanation(X_test_scaled, X_train_scaled)


In [9]:
# Initialize the FeatureTokenizer
num_features = X_train.shape[1]
embed_dim = 20
feature_tokenizer = FeatureTokenizer(num_features, embed_dim)

# Tokenize and encode inputs
X_train_encoded = feature_tokenizer(X_train_scaled)
X_test_encoded = feature_tokenizer(X_test_scaled)

# Initialize the model with uncertainty
input_dim = num_features
hidden_dim = 64
dropout_rate = 0.1

model_with_uncertainty = ModelWithUncertainty(input_dim, hidden_dim, dropout_rate, method='mc_dropout')

# Compile the model
model_with_uncertainty.compile(optimizer='adam', loss='mse')

# Train the model
model_with_uncertainty.fit(X_train_encoded, y_train, epochs=20, batch_size=32, validation_split=0.1)

# Make predictions and explanations
predictions_mean, predictions_std, lime_explanations, shap_explanations = model_with_uncertainty.predict_with_exp(X_test_encoded, X_train_encoded, num_samples=10)

# Print results
print("Predictions mean:", predictions_mean)
print("Predictions std:", predictions_std)

# Example of visualizing LIME and SHAP explanations
lime_explanation = lime_explanations[0]
lime_explanation.show_in_notebook()

shap.initjs()
shap.force_plot(shap_explanations[0], matplotlib=True)

Epoch 1/20


ValueError: Exception encountered when calling FeatureTokenizer.call().

[1mDimensions must be equal, but are 20 and 13 for '{{node model_with_uncertainty_1/feature_tokenizer_1_1/mul}} = Mul[T=DT_FLOAT](model_with_uncertainty_1/feature_tokenizer_1_1/strided_slice, model_with_uncertainty_1/feature_tokenizer_1_1/mul/ReadVariableOp)' with input shapes: [?,13,20,1], [13,64].[0m

Arguments received by FeatureTokenizer.call():
  • inputs=tf.Tensor(shape=(None, 13, 20), dtype=float32)

In [10]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, MultiHeadAttention
from tensorflow.keras import Sequential
from lime.lime_tabular import LimeTabularExplainer
import shap

class FeatureTokenizer(tf.keras.layers.Layer):
    def __init__(self, num_features, embed_dim):
        super(FeatureTokenizer, self).__init__()
        self.embed_dim = embed_dim
        self.feature_embeddings = tf.keras.layers.Embedding(num_features, embed_dim)
        
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        embeddings = []
        for i in range(inputs.shape[1]):
            feature_values = inputs[:, i]
            feature_embedding = self.feature_embeddings(tf.fill([batch_size], i))
            embeddings.append(feature_embedding * tf.expand_dims(feature_values, axis=-1))
        return tf.stack(embeddings, axis=1)

class MLPModelWithUncertainty(tf.keras.Model):
    def __init__(self, num_features, embed_dim, hidden_dim, dropout_rate, method='mc_dropout'):
        super(MLPModelWithUncertainty, self).__init__()
        self.method = method
        self.tokenizer = FeatureTokenizer(num_features, embed_dim)
        
        self.model = Sequential([
            Dense(hidden_dim, activation='relu'),
            Dropout(dropout_rate),
            Dense(hidden_dim, activation='relu'),
            Dropout(dropout_rate),
            Dense(1)
        ])
        
    def call(self, inputs, training=False):
        tokenized_inputs = self.tokenizer(inputs)
        tokenized_inputs = tf.reshape(tokenized_inputs, (tf.shape(tokenized_inputs)[0], -1))
        return self.model(tokenized_inputs, training=training)
    
    def predict_with_uncertainty(self, inputs, num_samples=10):
        predictions = []
        for _ in range(num_samples):
            predictions.append(self(inputs, training=True if self.method == 'mc_dropout' else False))
        predictions = tf.stack(predictions, axis=0)
        pred_mean = tf.reduce_mean(predictions, axis=0)
        pred_std = tf.math.reduce_std(predictions, axis=0)
        return pred_mean, pred_std
    
    def predict_with_exp(self, inputs, X_train, num_samples=10):
        pred_mean, pred_std = self.predict_with_uncertainty(inputs, num_samples)
        
        explainer = LimeTabularExplainer(X_train, mode='regression')
        lime_explanations = [explainer.explain_instance(inputs[i], self.predict_with_uncertainty, num_features=inputs.shape[1]) for i in range(inputs.shape[0])]
        
        shap_explainer = shap.KernelExplainer(self.predict_with_uncertainty, X_train)
        shap_explanations = [shap_explainer.shap_values(inputs[i]) for i in range(inputs.shape[0])]
        
        return pred_mean, pred_std, lime_explanations, shap_explanations

# Load and preprocess data
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

cali_housing_path = '../data/California_Houses.csv'
RANDOM_SEED = 492
cali_df = pd.read_csv(cali_housing_path)
y = cali_df['Median_House_Value']
X = cali_df.drop(columns=['Median_House_Value'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize the model with uncertainty
num_features = X_train_scaled.shape[1]
embed_dim = 20
hidden_dim = 64
dropout_rate = 0.1

model_with_uncertainty = MLPModelWithUncertainty(num_features, embed_dim, hidden_dim, dropout_rate, method='mc_dropout')

# Compile the model
model_with_uncertainty.compile(optimizer='adam', loss='mse')

# Train the model
model_with_uncertainty.fit(X_train_scaled, y_train, epochs=20, batch_size=32, validation_split=0.1)

# Make predictions and explanations
predictions_mean, predictions_std, lime_explanations, shap_explanations = model_with_uncertainty.predict_with_exp(X_test_scaled, X_train_scaled, num_samples=10)

# Print results
print("Predictions mean:", predictions_mean.numpy())
print("Predictions std:", predictions_std.numpy())

# Example of visualizing LIME and SHAP explanations
lime_explanation = lime_explanations[0]
lime_explanation.show_in_notebook()

shap.initjs()
shap.force_plot(shap_explanations[0], matplotlib=True)


Epoch 1/20


ValueError: Exception encountered when calling MLPModelWithUncertainty.call().

[1mShapes used to initialize variables must be fully-defined (no `None` dimensions). Received: shape=(None, 64) for variable path='mlp_model_with_uncertainty/sequential_2/dense_7/kernel'[0m

Arguments received by MLPModelWithUncertainty.call():
  • inputs=tf.Tensor(shape=(None, 13), dtype=float32)
  • training=True